### Flip horizontal, Rotate 15°, Brightness adjustment (+20%)

In [ ]:
import os
import cv2
import shutil
import albumentations as A
from tqdm import tqdm

# Fungsi untuk membaca file label YOLO
def read_yolo_label(label_path):
    boxes = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls = int(float(parts[0]))  # Konversi class ID (misalnya 1.0 → 1)
            bbox = list(map(float, parts[1:]))  # Ambil koordinat bbox
            boxes.append((cls, *bbox))  # Gabungkan class + bbox
    return boxes

# Fungsi untuk menyimpan file label YOLO
def save_yolo_label(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            # Format: class x_center y_center width height (format YOLO)
            line = f"{int(box[0])} {' '.join([f'{x:.6f}' for x in box[1:]])}\n"
            f.write(line)

# Definisikan pipeline augmentasi dari Albumentations
transform = A.Compose([
    A.HorizontalFlip(p=0.5),  # Flip horizontal dengan probabilitas 50%
    A.Rotate(limit=15, p=0.5),  # Rotasi gambar max 15 derajat
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5)  # Ubah brightness/contrast
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))  # Format bounding box dan label

# Konfigurasi path dan parameter augmentasi
input_base = "dataset_75_15_10"  # Direktori input
output_base = "dataset_75_15_10_augmented_2x"  # Direktori output hasil augmentasi
target_size = (640, 640)  # Ukuran target resize
augment_times = 2  # Jumlah augmentasi per gambar
splits = ["train", "valid", "test"]  # Daftar folder split data

# Loop untuk semua split (train/valid/test)
for split in splits:
    print(f"\n🔧 Processing split: {split}")

    # Path input dan output untuk image dan label
    input_img_dir = os.path.join(input_base, split, "images")
    input_lbl_dir = os.path.join(input_base, split, "labels")
    output_img_dir = os.path.join(output_base, split, "images")
    output_lbl_dir = os.path.join(output_base, split, "labels")

    # Buat folder output jika belum ada
    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_lbl_dir, exist_ok=True)

    # Ambil semua file gambar dari folder input
    image_files = [f for f in os.listdir(input_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Loop semua file gambar
    for fname in tqdm(image_files, desc=f"{split} images"):
        img_path = os.path.join(input_img_dir, fname)
        lbl_path = os.path.join(input_lbl_dir, os.path.splitext(fname)[0] + ".txt")

        # Jika tidak ada file label, skip
        if not os.path.exists(lbl_path):
            continue

        # Baca gambar
        image = cv2.imread(img_path)
        if image is None:
            print(f"⚠️ Gagal membaca: {img_path}")
            continue

        # Resize gambar ke ukuran target
        image = cv2.resize(image, target_size)

        # Baca label YOLO
        boxes = read_yolo_label(lbl_path)
        if not boxes:
            continue  # Skip jika tidak ada bbox

        # Simpan gambar dan label asli (yang sudah di-resize)
        output_img_path = os.path.join(output_img_dir, fname)
        output_lbl_path = os.path.join(output_lbl_dir, os.path.splitext(fname)[0] + ".txt")
        cv2.imwrite(output_img_path, image)
        save_yolo_label(output_lbl_path, boxes)

        # Pisahkan bbox dan class untuk transformasi
        bboxes = [box[1:] for box in boxes]  # Ambil hanya bbox-nya
        class_labels = [int(box[0]) for box in boxes]  # Ambil hanya class-nya

        # Lakukan augmentasi sebanyak `augment_times`
        for i in range(augment_times):
            transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            aug_img = transformed["image"]
            aug_boxes = transformed["bboxes"]
            aug_labels = transformed["class_labels"]

            # Simpan hasil augmentasi
            base_name = os.path.splitext(fname)[0]
            aug_img_name = f"{base_name}_aug{i}.jpg"
            aug_lbl_name = f"{base_name}_aug{i}.txt"

            cv2.imwrite(os.path.join(output_img_dir, aug_img_name), aug_img)
            save_yolo_label(
                os.path.join(output_lbl_dir, aug_lbl_name),
                [(int(cls), *bbox) for cls, bbox in zip(aug_labels, aug_boxes)]
            )

    print(f"✅ {split}: resize asli + {augment_times}x augmentasi selesai.")

# Menampilkan ringkasan jumlah gambar dan label setelah augmentasi
print("\n📊 Ringkasan jumlah dataset setelah augmentasi:")
total_imgs, total_labels = 0, 0

for split in splits:
    img_dir = os.path.join(output_base, split, "images")
    lbl_dir = os.path.join(output_base, split, "labels")

    # Hitung jumlah file gambar dan label di masing-masing folder
    num_imgs = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    num_labels = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])

    total_imgs += num_imgs
    total_labels += num_labels

    print(f"📁 {split}: {num_imgs} gambar, {num_labels} label")

# Total keseluruhan
print(f"\n📦 Total: {total_imgs} gambar, {total_labels} label")


🔧 Processing split: train


train images:   0%|          | 0/1540 [00:00<?, ?it/s]

train images: 100%|██████████| 1540/1540 [03:05<00:00,  8.29it/s]


✅ train: resize asli + 2x augmentasi selesai.

🔧 Processing split: valid


valid images: 100%|██████████| 308/308 [00:36<00:00,  8.38it/s]


✅ valid: resize asli + 2x augmentasi selesai.

🔧 Processing split: test


test images: 100%|██████████| 205/205 [00:25<00:00,  8.03it/s]

✅ test: resize asli + 2x augmentasi selesai.

📊 Ringkasan jumlah dataset setelah augmentasi:
📁 train: 4620 gambar, 4620 label
📁 valid: 924 gambar, 924 label
📁 test: 615 gambar, 615 label

📦 Total: 6159 gambar, 6159 label


###  Simpan gambar original, +1 gambar hasil flip horizontal, +1 gambar hasil flip vertical

In [ ]:
import os
import cv2
import shutil
import albumentations as A
from tqdm import tqdm

# Fungsi bantu baca dan simpan label YOLO
def read_yolo_label(label_path):
    boxes = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls = int(float(parts[0]))  # konversi aman jika ada float
            bbox = list(map(float, parts[1:]))
            boxes.append((cls, *bbox))
    return boxes

def save_yolo_label(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            line = f"{int(box[0])} {' '.join([f'{x:.6f}' for x in box[1:]])}\n"
            f.write(line)

# Konfigurasi path dan transformasi
input_base = "datasets/datalabelstudio_75_15_10_clean"
output_base = "datasets/datalabelstudio_75_15_10_clean_flip"
target_size = (640, 640)
splits = ["train", "valid", "test"]

# Definisi transformasi
flip_h_transform = A.Compose([
    A.HorizontalFlip(p=1.0)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

flip_v_transform = A.Compose([
    A.VerticalFlip(p=1.0)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Untuk menghitung total
summary = {}

for split in splits:
    print(f"\n🔧 Processing split: {split}")

    input_img_dir = os.path.join(input_base, split, "images")
    input_lbl_dir = os.path.join(input_base, split, "labels")
    output_img_dir = os.path.join(output_base, split, "images")
    output_lbl_dir = os.path.join(output_base, split, "labels")

    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_lbl_dir, exist_ok=True)

    image_files = [f for f in os.listdir(input_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    original_count = 0

    for fname in tqdm(image_files, desc=f"{split} images"):
        img_path = os.path.join(input_img_dir, fname)
        lbl_path = os.path.join(input_lbl_dir, os.path.splitext(fname)[0] + ".txt")

        if not os.path.exists(lbl_path):
            continue

        image = cv2.imread(img_path)
        if image is None:
            print(f"⚠️  Gagal membaca gambar: {img_path}")
            continue

        image = cv2.resize(image, target_size)
        boxes = read_yolo_label(lbl_path)
        if not boxes:
            continue

        bboxes = [box[1:] for box in boxes]
        class_labels = [int(box[0]) for box in boxes]  # pastikan int

        # Simpan versi resized dari gambar original
        cv2.imwrite(os.path.join(output_img_dir, fname), image)
        save_yolo_label(os.path.join(output_lbl_dir, os.path.splitext(fname)[0] + ".txt"), boxes)
        original_count += 1

        # Flip Horizontal
        flipped_h = flip_h_transform(image=image, bboxes=bboxes, class_labels=class_labels)
        fliph_labels = [(int(cls), *bbox) for cls, bbox in zip(flipped_h["class_labels"], flipped_h["bboxes"])]
        cv2.imwrite(os.path.join(output_img_dir, f"{os.path.splitext(fname)[0]}_fliph.jpg"), flipped_h["image"])
        save_yolo_label(os.path.join(output_lbl_dir, f"{os.path.splitext(fname)[0]}_fliph.txt"), fliph_labels)

        # Flip Vertical
        flipped_v = flip_v_transform(image=image, bboxes=bboxes, class_labels=class_labels)
        flipv_labels = [(int(cls), *bbox) for cls, bbox in zip(flipped_v["class_labels"], flipped_v["bboxes"])]
        cv2.imwrite(os.path.join(output_img_dir, f"{os.path.splitext(fname)[0]}_flipv.jpg"), flipped_v["image"])
        save_yolo_label(os.path.join(output_lbl_dir, f"{os.path.splitext(fname)[0]}_flipv.txt"), flipv_labels)

    total_output = original_count * 3
    summary[split] = (original_count, total_output)
    print(f"✅ {split}: {original_count} gambar asli → {total_output} total (termasuk flip)")

# Cetak ringkasan akhir
print("\n📊 Ringkasan Dataset:")
for split, (ori, total) in summary.items():
    print(f"🗂 {split.capitalize()}: {ori} gambar asli → {total} total gambar")


C:\Users\alimu\AppData\Roaming\Python\Python310\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()



🔧 Processing split: train


train images: 100%|██████████| 1401/1401 [02:21<00:00,  9.88it/s]


✅ train: 1401 gambar asli → 4203 total (termasuk flip)

🔧 Processing split: valid


valid images: 100%|██████████| 272/272 [00:28<00:00,  9.66it/s]


✅ valid: 272 gambar asli → 816 total (termasuk flip)

🔧 Processing split: test


test images: 100%|██████████| 190/190 [00:19<00:00,  9.64it/s]

✅ test: 190 gambar asli → 570 total (termasuk flip)

📊 Ringkasan Dataset:
🗂 Train: 1401 gambar asli → 4203 total gambar
🗂 Valid: 272 gambar asli → 816 total gambar
🗂 Test: 190 gambar asli → 570 total gambar


: 